# HW10 – Improving RAG with Reranking, Chunking and Conversational Memory
**Dataset:** `ccdv/arxiv-summarization` (document config), first 100 train docs · **Vector DB:** ChromaDB · **Embedder:** all-MiniLM-L6-v2 · **Reranker:** ms-marco-MiniLM-L-6-v2

Pipeline: preprocess → 2 chunking strategies → embed + index → retrieve (dense) → rerank → conversational memory → evaluate (Precision@K, Recall@K, MRR) → compare.

## 0. Setup & data

In [1]:
# !pip install chromadb sentence-transformers datasets pandas

import json, re, os, sys, time
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

ROOT = os.getcwd()
os.environ.setdefault("HF_HOME", os.path.join(ROOT, ".hf"))
EMB_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CHUNK_WORDS, OVERLAP_WORDS = 150, 30
N_CANDIDATES = 20          # retrieved from vector DB before reranking
KS = (1, 3, 5, 10)

C:\Users\sivasurya\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1109 18:04:05.552000 69224 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
import os, json
if not os.path.exists("docs100.json"):
    from datasets import load_dataset
    ds = load_dataset("ccdv/arxiv-summarization", "document", split="train", streaming=True)  # streaming: avoids multi-GB download
    docs_ = [{"id": i, "article": x["article"], "abstract": x["abstract"]} for i, x in zip(range(100), ds)]
    json.dump(docs_, open("docs100.json", "w"))
docs = json.load(open("docs100.json"))
print(len(docs), "documents; avg words =", int(np.mean([len(d["article"].split()) for d in docs])))

100 documents; avg words = 6516


## 1. Preprocessing

In [3]:
def preprocess(text: str):
    """Return a list of clean sentences. The dataset stores one sentence per line,
    with LaTeX placeholders (@xmath12, @xcite) and ' .' tokenisation artefacts."""
    text = re.sub(r"@xcite", "", text)
    text = re.sub(r"@xmath\d+", "[math]", text)
    text = re.sub(r"(\[math\]\s*){2,}", "[math] ", text)       # collapse runs
    sents = []
    for line in text.split("\n"):
        line = re.sub(r"\s+", " ", line).strip()
        line = re.sub(r"\s+([.,;:!?)])", r"\1", line)          # ' .' -> '.'
        line = re.sub(r"\(\s+", "(", line)
        # lines can be whole paragraphs -> split on sentence-final punctuation
        for s in re.split(r"(?<=[.?!])\s+(?=[A-Za-z\[(\"'])", line):
            if len(s.split()) >= 3:                             # drop fragments
                sents.append(s.strip())
    return sents

In [4]:
print("RAW  :", docs[0]["article"][:300].replace("\n", " "))
print("CLEAN:", " ".join(preprocess(docs[0]["article"])[:2])[:300])

RAW  : additive models @xcite provide an important family of models for semiparametric regression or classification . some reasons for the success of additive models are their increased flexibility when compared to linear or generalized linear models and their increased interpretability when compared to fu
CLEAN: additive models provide an important family of models for semiparametric regression or classification. some reasons for the success of additive models are their increased flexibility when compared to linear or generalized linear models and their increased interpretability when compared to fully nonp


## 2. Two chunking strategies
1. **fixed window** – 150 words, 30 overlap (cuts mid-sentence)
2. **sentence-aware** – whole sentences packed to ~150 words, 1-sentence overlap

In [5]:
def chunk_fixed(sents, size=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    """Strategy 1: fixed-size sliding word window (cuts mid-sentence)."""
    words = " ".join(sents).split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

def chunk_sentence(sents, size=CHUNK_WORDS, overlap_sents=1):
    """Strategy 2: sentence-aware packing up to ~size words, carrying the last
    sentence into the next chunk as overlap. Never splits a sentence."""
    chunks, cur, n = [], [], 0
    flat = []                       # hard-split run-on "sentences" (tables, ref lists)
    for s in sents:
        ws = s.split()
        flat += [" ".join(ws[i:i + size]) for i in range(0, len(ws), size)] if len(ws) > size else [s]
    for s in flat:
        w = len(s.split())
        if cur and n + w > size:
            chunks.append(" ".join(cur))
            cur = cur[-overlap_sents:] if overlap_sents else []
            n = sum(len(x.split()) for x in cur)
        cur.append(s); n += w
    if cur:
        chunks.append(" ".join(cur))
    return chunks

CHUNKERS = {"fixed_window": chunk_fixed, "sentence_aware": chunk_sentence}

## 3. Embed and index in ChromaDB

In [6]:
def build_index(docs, embedder, client):
    cols, corpus = {}, {}
    for name, fn in CHUNKERS.items():
        recs = []
        for d in docs:
            for j, c in enumerate(fn(preprocess(d["article"]))):
                recs.append({"id": f"{name}-d{d['id']}-c{j}", "doc": d["id"], "text": c})
        corpus[name] = recs
        try: client.delete_collection(name)
        except Exception: pass
        col = client.create_collection(name, metadata={"hnsw:space": "cosine"})
        embs = embedder.encode([r["text"] for r in recs], batch_size=64,
                               normalize_embeddings=True, show_progress_bar=False)
        for i in range(0, len(recs), 2000):
            b = recs[i:i + 2000]
            col.add(ids=[r["id"] for r in b], embeddings=embs[i:i + 2000].tolist(),
                    documents=[r["text"] for r in b],
                    metadatas=[{"doc": r["doc"]} for r in b])
        cols[name] = col
        wc = [len(r["text"].split()) for r in recs]
        print(f"[index] {name:15s} chunks={len(recs):5d} mean_words={np.mean(wc):.0f} "
              f"min={min(wc)} max={max(wc)}")
    return cols, corpus

## 4. Retrieval (dense) and reranking (cross-encoder)

In [7]:
class Retriever:
    def __init__(self, cols, embedder, reranker):
        self.cols, self.emb, self.rr = cols, embedder, reranker

    def search(self, query, chunking, k=10, rerank=False):
        q = self.emb.encode([query], normalize_embeddings=True)[0].tolist()
        res = self.cols[chunking].query(query_embeddings=[q], n_results=N_CANDIDATES)
        hits = [{"id": i, "text": t, "doc": m["doc"], "score": 1 - d}
                for i, t, m, d in zip(res["ids"][0], res["documents"][0],
                                      res["metadatas"][0], res["distances"][0])]
        if rerank:
            s = self.rr.predict([(query, h["text"]) for h in hits])
            for h, sc in zip(hits, s): h["score"] = float(sc)
            hits.sort(key=lambda h: -h["score"])
        return hits[:k]

## 5. Conversational memory
A follow-up is rewritten as `<previous question> <current question>` before retrieval/reranking.

In [8]:
class ConversationMemory:
    """Rolling window of previous turns. A follow-up is rewritten by prefixing the
    previous user question(s), so pronouns/ellipsis ('they', 'it', 'that') are
    resolved by the retriever through the earlier topical context."""
    def __init__(self, window=1):
        self.window, self.turns = window, []

    def contextualise(self, query):
        if not self.turns:
            return query
        prev = " ".join(t["query"] for t in self.turns[-self.window:])
        return f"{prev} {query}"

    def add(self, query, hits):
        self.turns.append({"query": query, "top_doc": hits[0]["doc"] if hits else None})

## 6. Evaluation setup
Relevance rubric: chunk is relevant iff it is from the target document **and** mentions ≥3 (D6: ≥2) of the hand-chosen concepts needed to answer the query. Queries S1–S8 use paper vocabulary; S9–S14 are paraphrased (harder). D1–D6 are two-turn dialogues whose 2nd turn is unanswerable alone.

In [9]:
def is_relevant(text, doc, spec):
    """Rubric (documented in report): chunk is relevant iff it comes from a target
    document AND mentions >= spec['min_hits'] distinct topical concepts needed to
    answer the question, i.e. it is useful evidence for the answer."""
    if doc not in spec["docs"]:
        return False
    t = text.lower()
    return sum(bool(re.search(p, t)) for p in spec["terms"]) >= spec["min_hits"]

def metrics(hits, spec, n_rel_total, ks=KS):
    rel = [is_relevant(h["text"], h["doc"], spec) for h in hits]
    out = {}
    for k in ks:
        r = sum(rel[:k])
        out[f"P@{k}"] = r / k
        out[f"R@{k}"] = r / n_rel_total if n_rel_total else 0.0
    rr = next((1 / (i + 1) for i, x in enumerate(rel) if x), 0.0)
    out["RR"] = rr
    return out

QUERIES = [  # single-turn queries ----------------------------------------------
 dict(id="S1", q="What learning rates are obtained for regularized kernel based methods for additive models?",
      docs=[0], terms=[r"learning rate", r"additive", r"kernel", r"regulari"], min_hits=3),
 dict(id="S2", q="How does synchronization occur in coupled quadratic maps?",
      docs=[36], terms=[r"synchroniz", r"coupl", r"quadratic|logistic", r"map"], min_hits=3),
 dict(id="S3", q="How can squeezed vacuum be used for non-invasive measurement of cavity parameters?",
      docs=[33], terms=[r"squeez", r"cavity", r"measur", r"vacuum"], min_hits=3),
 dict(id="S4", q="What causes the enhanced mid-infrared second harmonic generation in SiC nanopillars?",
      docs=[6], terms=[r"second.harmonic|shg", r"nanopillar", r"resonan", r"sic"], min_hits=3),
 dict(id="S5", q="How is bend loss predicted for large mode area photonic crystal fibers?",
      docs=[37], terms=[r"bend", r"loss", r"photonic crystal|pcf", r"mode area|lma"], min_hits=3),
 dict(id="S6", q="Why does magnetoresistance become linear in the surface state of a topological insulator?",
      docs=[20], terms=[r"magnetoresistance", r"linear", r"topological insulator", r"surface state"], min_hits=3),
 dict(id="S7", q="How do phylogenetic networks model reticulate evolution?",
      docs=[58], terms=[r"phylogenetic", r"network", r"reticulat", r"evolution"], min_hits=3),
 dict(id="S8", q="What was observed during the 1994 outburst of the T Tauri star EX Lupi?",
      docs=[48], terms=[r"ex lup", r"outburst", r"t tauri", r"spectr|photomet"], min_hits=3),
 # harder: paraphrased, little exact jargon overlap, many topically-near documents
 dict(id="S9", q="How is the spin structure of the proton and neutron probed through form factors?",
      docs=[4], terms=[r"spin", r"nucleon", r"form factor", r"quark|gluon"], min_hits=3),
 dict(id="S10", q="What do microwave background anisotropy maps tell us about open universe models?",
      docs=[12], terms=[r"open", r"dmr|cobe|microwave", r"normali[sz]ation|amplitude|quadrupole", r"model"], min_hits=3),
 dict(id="S11", q="How are signals from merging binary stars found in noisy interferometer data using templates?",
      docs=[27], terms=[r"matched.filter|template", r"gravitational", r"compact|binary|merger|inspiral", r"detector|noise|data"], min_hits=3),
 dict(id="S12", q="How does hot asymmetric nuclear matter separate into liquid and vapour phases?",
      docs=[49], terms=[r"liquid.gas|phase transition", r"neutron.rich|asymmetr", r"nuclear matter", r"temperature|thermal|spinodal"], min_hits=3),
 dict(id="S13", q="Why does a chaotic ratchet move particles in the opposite direction?",
      docs=[2], terms=[r"ratchet", r"current", r"revers", r"chaotic|bifurcation"], min_hits=3),
 dict(id="S14", q="How can online astronomy resources be made usable for people with disabilities?",
      docs=[40], terms=[r"access", r"librar", r"disab|impair|challenged|blind", r"web|online|resource"], min_hits=3),
]

DIALOGS = [  # multi-turn: turn 2 is meaningless without turn 1 -------------------
 dict(id="D1", turns=["What learning rates are obtained for regularized kernel based methods for additive models?",
                      "Why do they compare favourably in high dimensions?"],
      spec2=dict(docs=[0], terms=[r"high.dimension", r"learning rate|rate", r"additive", r"nonparametric|optimal"], min_hits=3)),
 dict(id="D2", turns=["How does synchronization occur in coupled quadratic maps?",
                      "What conditions make it stable?"],
      spec2=dict(docs=[36], terms=[r"stab", r"synchroniz", r"condition|threshold|critical", r"coupl"], min_hits=3)),
 dict(id="D3", turns=["What was observed during the 1994 outburst of the T Tauri star EX Lupi?",
                      "How bright did it get and how long did it last?"],
      spec2=dict(docs=[48], terms=[r"outburst", r"mag|bright", r"days?|weeks?|months?|duration|decline", r"ex lup"], min_hits=3)),
 dict(id="D4", turns=["How is bend loss predicted for large mode area photonic crystal fibers?",
                      "How accurate is it compared with numerical simulations?"],
      spec2=dict(docs=[37], terms=[r"accura|agree|good", r"numerical|simulation|multipole|full.vector", r"bend", r"loss"], min_hits=3)),
 dict(id="D5", turns=["Why does magnetoresistance become linear in the surface state of a topological insulator?",
                      "What is the role of the Dirac cone there?"],
      spec2=dict(docs=[20], terms=[r"dirac", r"linear", r"magnetoresistance", r"surface"], min_hits=3)),
 dict(id="D6", turns=["How do phylogenetic networks model reticulate evolution?",
                      "How can they be visualised?"],
      spec2=dict(docs=[58], terms=[r"visuali[sz]", r"network", r"phylogenetic|reticulat|evolution"], min_hits=2)),
]

def n_relevant(corpus_recs, spec):
    return sum(is_relevant(r["text"], r["doc"], spec) for r in corpus_recs)

def avg(rows):
    return {k: float(np.mean([r[k] for r in rows])) for k in rows[0]}

def snippet(t, n=170): return t[:n].replace("\n", " ") + ("..." if len(t) > n else "")

## 7. Build the indexes

In [10]:
embedder = SentenceTransformer(EMB_NAME)
reranker = CrossEncoder(RERANK_NAME)
client = chromadb.PersistentClient(path=os.path.join(ROOT, "chroma_db"))
cols, corpus = build_index(docs, embedder, client)
R = Retriever(cols, embedder, reranker)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3316.78it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3112.45it/s]

[index] fixed_window    chunks= 4724 mean_words=149 min=31 max=150


[index] sentence_aware  chunks= 5256 mean_words=136 min=9 max=300


## 8. Sample queries: without vs with reranking (top-3, ✔ = relevant)

In [11]:
def show(qs, chunking):
    print("="*110); print(f"[{qs['id']}] {qs['q']}   (chunking = {chunking})")
    for rr in (False, True):
        print(f"  --- {'WITH reranking' if rr else 'WITHOUT reranking'}")
        for i, h in enumerate(R.search(qs["q"], chunking, k=3, rerank=rr), 1):
            print(f"   {i}. doc {h['doc']:>2} score {h['score']:7.3f} {'✔' if is_relevant(h['text'], h['doc'], qs) else '✘'}  {snippet(h['text'], 150)}")
for qs in [QUERIES[1], QUERIES[2], QUERIES[13]]:
    for ch in CHUNKERS: show(qs, ch)

[S2] How does synchronization occur in coupled quadratic maps?   (chunking = fixed_window)
  --- WITHOUT reranking
   1. doc 36 score   0.695 ✔  will show, an arbitrarily large number of orbits may become stable. the existence of a large number of stable periodic orbits for just two coupled qua...
   2. doc 36 score   0.637 ✘  is said to synchronize if [math] and if all orbits synchronize, then we say to have synchronization of the map. synchronization is the simplest dynami...
   3. doc 36 score   0.573 ✘  prove that synchronization occurs. when [math] is sufficiently small, this happens even though [math]. starting from synchronization and modifying the...
  --- WITH reranking
   1. doc 36 score   3.881 ✔  will show, an arbitrarily large number of orbits may become stable. the existence of a large number of stable periodic orbits for just two coupled qua...
   2. doc 36 score   2.640 ✘  is said to synchronize if [math] and if all orbits synchronize, then we say to have synchronizatio

   1. doc 36 score   0.708 ✔  by coupling, as we will show, an arbitrarily large number of orbits may become stable. the existence of a large number of stable periodic orbits for j...
   2. doc 36 score   0.687 ✔  rather than merely focusing on multistability, we also study the phenomenology of two coupled quadratic maps, in particular the bifurcations of period...
   3. doc 36 score   0.617 ✘  finally, note that the dynamics commutes with the symmetry [math] or [math] in the original variables. if [math], then [math] and [math]. in this case...
  --- WITH reranking
   1. doc 36 score   4.653 ✔  rather than merely focusing on multistability, we also study the phenomenology of two coupled quadratic maps, in particular the bifurcations of period...
   2. doc 36 score   4.494 ✔  by coupling, as we will show, an arbitrarily large number of orbits may become stable. the existence of a large number of stable periodic orbits for j...
   3. doc 36 score   1.184 ✘  the condition [math] is not n

   1. doc 40 score   0.646 ✘  a fair number of astronomers and astronomy students have a physical challenge. it is our responsibility to learn the basics of accessibility to be abl...
   2. doc 40 score   0.596 ✔  in theory, it is possible for almost anyone to access online information, but in practice, this requires know - how and proper tools. plenty of assist...
   3. doc 40 score   0.516 ✘  an astronomer who is able - bodied today could have accessibility issues tomorrow. we can not expect that she or he is willing to give up practicing s...
  --- WITH reranking
   1. doc 40 score  -1.883 ✔  in theory, it is possible for almost anyone to access online information, but in practice, this requires know - how and proper tools. plenty of assist...
   2. doc 40 score  -5.552 ✘  a fair number of astronomers and astronomy students have a physical challenge. it is our responsibility to learn the basics of accessibility to be abl...
   3. doc 40 score  -5.651 ✔  one can produce solid researc

## 9. Evaluation: without vs with reranking, chunking 1 vs 2

In [12]:
import pandas as pd
rows, perq = [], []
for ch in CHUNKERS:
    for rr in (False, True):
        ms = []
        for qs in QUERIES:
            tot = n_relevant(corpus[ch], qs)
            m = metrics(R.search(qs["q"], ch, k=10, rerank=rr), qs, tot); ms.append(m)
            perq.append({"query": qs["id"], "chunking": ch, "rerank": rr, "RR": m["RR"], "P@5": m["P@5"], "n_relevant_chunks": tot})
        rows.append({"chunking": ch, "rerank": rr, **avg(ms)})
single = pd.DataFrame(rows)
cols_ = ["chunking","rerank","P@1","P@3","P@5","P@10","R@5","R@10","RR"]
single[cols_].rename(columns={"RR":"MRR"}).round(3)

,chunking,rerank,P@1,P@3,P@5,P@10,R@5,R@10,MRR
0,fixed_window,False,0.857,0.738,0.700,0.586,0.451,0.685,0.911
1,fixed_window,True,1.000,0.810,0.757,0.643,0.516,0.751,1.000
2,sentence_aware,False,0.857,0.810,0.757,0.629,0.434,0.668,0.917
3,sentence_aware,True,0.929,0.786,0.743,0.664,0.408,0.713,0.964


In [13]:
pq = pd.DataFrame(perq)
pq.pivot_table(index="query", columns=["chunking","rerank"], values="RR").round(2).loc[[q["id"] for q in QUERIES]]  # per-query reciprocal rank

chunking fixed_window       sentence_aware      
rerank          False True           False True 
query                                           
S1               1.00   1.0           1.00   1.0
S2               1.00   1.0           1.00   1.0
S3               1.00   1.0           1.00   1.0
S4               1.00   1.0           1.00   0.5
S5               1.00   1.0           1.00   1.0
S6               1.00   1.0           1.00   1.0
S7               1.00   1.0           1.00   1.0
S8               1.00   1.0           1.00   1.0
S9               1.00   1.0           1.00   1.0
S10              1.00   1.0           1.00   1.0
S11              0.50   1.0           0.33   1.0
S12              1.00   1.0           1.00   1.0
S13              1.00   1.0           1.00   1.0
S14              0.25   1.0           0.50   1.0

## 10. Conversational memory – multi-turn demo (turn 2 depends on turn 1)

In [14]:
def run_dialog(dlg, ch, rr, use_mem, k=3):
    mem = ConversationMemory(window=1)
    for ti, q in enumerate(dlg["turns"]):
        query = mem.contextualise(q) if use_mem else q
        hits = R.search(query, ch, k=10, rerank=rr); mem.add(q, hits)
        if ti == 1:
            print(f"  Turn 1: {dlg['turns'][0]}\n  Turn 2: {q}\n  Query actually sent: {query}")
            for i, h in enumerate(hits[:k], 1):
                print(f"   {i}. doc {h['doc']:>2} {'✔' if is_relevant(h['text'], h['doc'], dlg['spec2']) else '✘'}  {snippet(h['text'], 140)}")
for d in [DIALOGS[0], DIALOGS[1]]:
    for use_mem in (False, True):
        print("="*100); print(f"{d['id']} – {'WITH memory' if use_mem else 'WITHOUT memory'} (sentence-aware + rerank)")
        run_dialog(d, "sentence_aware", True, use_mem)

D1 – WITHOUT memory (sentence-aware + rerank)
  Turn 1: What learning rates are obtained for regularized kernel based methods for additive models?
  Turn 2: Why do they compare favourably in high dimensions?
  Query actually sent: Why do they compare favourably in high dimensions?
   1. doc  0 ✔  we now add some theoretical and numerical comparisons on the goodness of our learning rates with those from the literature. as already menti...
   2. doc  0 ✔  the figures seem to indicate that our learning rate from theorem [ mainratesthm ] is probably not optimal for small dimensions. however, the...
   3. doc 46 ✘  this experience leads naturally to the idea of trying to define what the higher dimensional analogues of the gray tensor product are, so as ...
D1 – WITH memory (sentence-aware + rerank)
  Turn 1: What learning rates are obtained for regularized kernel based methods for additive models?
  Turn 2: Why do they compare favourably in high dimensions?
  Query actually sent: What learn

## 11. Evaluation: without vs with conversational memory (turn-2 queries, 6 dialogues)

In [15]:
rows = []
for ch in CHUNKERS:
    for rr in (False, True):
        for use_mem in (False, True):
            ms = []
            for dlg in DIALOGS:
                mem = ConversationMemory(window=1)
                for ti, q in enumerate(dlg["turns"]):
                    query = mem.contextualise(q) if use_mem else q
                    hits = R.search(query, ch, k=10, rerank=rr); mem.add(q, hits)
                    if ti == 1: ms.append(metrics(hits, dlg["spec2"], n_relevant(corpus[ch], dlg["spec2"])))
            rows.append({"chunking": ch, "rerank": rr, "memory": use_mem, **avg(ms)})
memdf = pd.DataFrame(rows)
memdf[["chunking","rerank","memory","P@1","P@3","P@5","R@5","R@10","RR"]].rename(columns={"RR":"MRR"}).round(3)

,chunking,rerank,memory,P@1,P@3,P@5,R@5,R@10,MRR
0,fixed_window,False,False,0.000,0.000,0.000,0.000,0.028,0.024
1,fixed_window,False,True,1.000,0.722,0.667,0.460,0.639,1.000
2,fixed_window,True,False,0.167,0.111,0.067,0.032,0.084,0.270
3,fixed_window,True,True,1.000,0.833,0.800,0.564,0.760,1.000
4,sentence_aware,False,False,0.167,0.111,0.067,0.039,0.088,0.246
5,sentence_aware,False,True,0.833,0.722,0.767,0.481,0.668,0.917
6,sentence_aware,True,False,0.167,0.222,0.167,0.090,0.118,0.347
7,sentence_aware,True,True,1.000,0.944,0.867,0.539,0.706,1.000


## 12. Findings

**Reranking vs none.** Reranking mainly improves the top of the list: with fixed chunks MRR 0.911 → 1.000, P@1 0.857 → 1.000, R@5 0.416 → 0.516; with sentence chunks MRR 0.917 → 0.964 and R@10 0.668 → 0.713 (P@5/R@5 flat to slightly lower). Gains come from the paraphrased queries (S11, S14: first relevant chunk moves to rank 1); on easy queries the dense retriever is already at MRR≈1, and reranking occasionally hurts (S4, sentence chunks).

**Chunking 1 vs 2.** Without reranking, sentence-aware chunks are better at early precision (P@3 0.810 vs 0.738, P@5 0.757 vs 0.686) because chunks are coherent units. With reranking, fixed windows score slightly higher on R@5/R@10/MRR. Differences are small relative to the noise of 14 queries; neither is a clear winner.

**Memory vs none.** Large and consistent: without memory the second turn ("Why do they…", "What conditions make it stable?") gets MRR 0.02–0.35 and P@5 ≤ 0.17; with memory MRR 0.92–1.00 and P@5 0.67–0.87. Best overall: sentence-aware + reranking + memory (P@5 0.867, MRR 1.0). Reranking and memory are complementary.

**Limitations.** Tiny hand-built evaluation (14 queries / 6 dialogues); relevance is a keyword rubric tied to one target document, not every chunk read by hand; recall denominators differ between chunkings; the memory test is favourable by design (follow-ups are unanswerable alone, no topic switches) and simple concatenation would fail after a topic change; only retrieval is evaluated (no LLM generation); candidate pool of 20 bounds what reranking can recover.